# --- import

In [ ]:
import itertools
import joblib
import json, time
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import shap
import ast

from catboost import CatBoostClassifier
from catboost import CatBoostClassifier, Pool
from copy import deepcopy
from lightgbm import LGBMClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from pathlib import Path
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_class_weight
from src.models.ML_models_tripl import *
from src.utils.data_utils import *
from src.utils.metrics import *
from xgboost import XGBClassifier



# ---basic parameters

In [ ]:
RANDOM_STATE = 42
ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

def fit_and_eval(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return compute_tp_f1_from_preds(y_val, preds), model

def save_best(model_name, ku, kd, hold, best_params, best_model):
    tag = f"ku{ku}_kd{kd}_hold{hold}"
    # hiperparametrs
    with open(ART_DIR / f"{model_name}_{tag}_best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)
    # model save
    if model_name == "RandomForest":
        import joblib
        joblib.dump(best_model, ART_DIR / f"{model_name}_{tag}.pkl")
    elif model_name == "LightGBM":
        # save booster in txt
        best_model.booster_.save_model(str(ART_DIR / f"{model_name}_{tag}.txt"))
    elif model_name == "XGBoost":
        best_model.save_model(str(ART_DIR / f"{model_name}_{tag}.json"))
    elif model_name == "CatBoost":
        best_model.save_model(str(ART_DIR / f"{model_name}_{tag}.cbm"))

In [ ]:
BASE = Path("artifacts")
RF_DIR = BASE / "rf"
RF_DIR.mkdir(parents=True, exist_ok=True)
LGBM_DIR = BASE / "lgbm"
LGBM_DIR.mkdir(parents=True, exist_ok=True)
XGB_DIR = BASE / "xgb"
XGB_DIR.mkdir(parents=True, exist_ok=True)
CAT_DIR = BASE / "cat"
CAT_DIR.mkdir(parents=True, exist_ok=True)

print("Folders created")


In [ ]:
df = pd.read_parquet("data/processed/BTC_USDT_1h_futures.parquet")

print("File read. Size:", df.shape)
df.head()


In [ ]:
# main parameters
KU = 6.0
KD = 2.0
HOLD = 336

#KU_list = [ 1.0, 2.0, 3.0, 4.0]
#KD_list = [0.5, 1.0, 1.5]
KU_list = [ 6.0]
KD_list = [ 2.0]

EXCLUDE_FEATURES = [
    "open", "high", "low", "close", "atr_200",
]

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(df, KU=KU, KD=KD, HOLD=HOLD)


In [ ]:
#tune_lightgbm_for_ku_kd(KU, KD, X_train, y_train, X_val, y_val)

In [ ]:
#tune_xgb_for_ku_kd(KU, KD, X_train, y_train, X_val, y_val)

In [ ]:
#tune_catboost_for_ku_kd(KU, KD, X_train, y_train, X_val, y_val)

In [ ]:
#tune_random_forest_for_ku_kd(KU, KD, X_train, y_train, X_val, y_val)

# --- Run All models

In [ ]:
df_results = run_full_binary_pipeline(
    df,
    KU_list=KU_list,
    KD_list=KD_list,
    HOLD=HOLD
)
df_results


# --- check the best combination of KU/KD based on lgbm

In [ ]:
def evaluate_trained_model_across_kukd(
    model,
    model_name: str,
    KU_lst,
    KD_lst,
    df,
    HOLD=HOLD
):
    """
    Checks one ALREADY TRAINED model on different KU/KD.
    - rebuilds y-labels for KU/KD
    - runs predict_proba without retraining
    - DL-threshold on validation
    - evaluation on the test set
    """

    rows = []

    for KU in KU_lst:
        for KD in KD_lst:

            print(f"\n======================")
            print(f"📌 MODEL={model_name}, KU={KU}, KD={KD}")
            print(f"======================")

            X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
                df, KU=KU, KD=KD, HOLD=HOLD, force=True
            )

            for d in [X_train, y_train, X_val, y_val, X_test, y_test]:
                d.reset_index(drop=True, inplace=True)

            proba_val = model.predict_proba(X_val)[:, 1]
            proba_test = model.predict_proba(X_test)[:, 1]

            best_thr, thr_metrics = dl_threshold_for_probs(
                proba_val, y_val, KU, KD
            )

            print(f"✔ Best threshold = {best_thr:.3f}")

            y_pred_test = (proba_test >= best_thr).astype(int)

            m_test = triple_barrier_metrics(
                y_true=y_test,
                y_pred=y_pred_test,
                p_all=proba_test,
                ku=KU, kd=KD
            )

            # 5) Stability (gap)
            macro_gap = abs(m_test["macro_f1"] - thr_metrics["macro_f1"])
            precision_gap = abs(m_test["tp_precision"] - thr_metrics["tp_precision"])
            bss_gap = abs(m_test["bss"] - thr_metrics["bss"])

            # 6) Save row
            rows.append({
                "KU": KU,
                "KD": KD,
                "model": model_name,
                "threshold": best_thr,

                # VAL metrics
                "macro_f1_val": thr_metrics["macro_f1"],
                "tp_precision_val": thr_metrics["tp_precision"],
                "tp_recall_val": thr_metrics["tp_recall"],
                "tp_f1_val": thr_metrics["tp_f1"],
                "bss_val": thr_metrics["bss"],

                # TEST metrics
                "macro_f1_test": m_test["macro_f1"],
                "tp_precision_test": m_test["tp_precision"],
                "tp_recall_test": m_test["tp_recall"],
                "tp_f1_test": m_test["tp_f1"],
                "bss_test": m_test["bss"],

                # stability
                "macro_gap": macro_gap,
                "precision_gap": precision_gap,
                "bss_gap": bss_gap,
            })

    return pd.DataFrame(rows)


In [ ]:
model = LGBMClassifier()
model = joblib.load("artifacts/lgbm/best_lgbm_ku6.0_kd2.0_hold336.pkl")

df_results_cat = evaluate_trained_model_across_kukd(
    model=model,
    model_name="LGBM",
    KU_lst=[1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6],
    KD_lst=[1, 1.5, 2, 2.5, 3],
    df=df,
    HOLD=HOLD
)


In [ ]:
def plot_metrics_heatmap_single_model(df_model):
    """
    Build heatmaps (2x2) for 4 metrics of a single model:
    Macro F1, TP Precision, TP Recall, BSS
    """

    metrics = [
        ("macro_f1_test", "Macro F1 (Test)"),
        ("tp_precision_test", "TP Precision (Test)"),
        ("tp_recall_test", "TP Recall (Test)"),
        ("bss_test", "Brier Skill Score (Test)"),
    ]

    available = [(col, title) for col, title in metrics if col in df_model.columns]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    for ax, (col, title) in zip(axes.flatten(), available):

        pivot = df_model.pivot(index="KU", columns="KD", values=col)

        sns.heatmap(
            pivot,
            annot=True,
            cmap="viridis",
            fmt=".3f",
            linewidths=0.5,
            ax=ax
        )

        ax.set_title(title, fontsize=16)
        ax.set_xlabel("KD")
        ax.set_ylabel("KU")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_metrics_heatmap_single_model(df_results_cat)

In [ ]:
KU_lst = [1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6]
KD_lst = [1, 1.5, 2, 2.5, 3]
HOLD = 336
base_dir = Path("artifacts/splits")

# ------------------------------
# 1. Collecting statistics
# ------------------------------
records = []

for KU in KU_lst:
    for KD in KD_lst:

        folder = base_dir / f"ku{KU}_kd{KD}_hold{HOLD}"
        train_p = folder / "train.parquet"

        if not train_p.exists():
            continue

        train_df = pd.read_parquet(train_p)

        if "y" not in train_df.columns:
            continue

        vc = train_df["y"].value_counts(normalize=True)

        records.append({
            "KU": KU,
            "KD": KD,
            "p_expiry": vc.get(0, 0),
            "p_sl": vc.get(1, 0),
            "p_tp": vc.get(2, 0),
        })

df_stats = pd.DataFrame(records)

# ------------------------------
# 2. Plot 3 heatmaps on one figure
# ------------------------------
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

heat_targets = [
    ("p_tp", "TP Share (class = 2)"),
    ("p_sl", "SL Share (class = 1)"),
    ("p_expiry", "Expiry Share (class = 0)")
]

for ax, (col, title) in zip(axes, heat_targets):
    pivot = df_stats.pivot(index="KU", columns="KD", values=col)
    sns.heatmap(
        pivot,
        annot=True,
        cmap="viridis",
        fmt=".2f",
        ax=ax
    )
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("KD")
    ax.set_ylabel("KU")

plt.tight_layout()
plt.show()


# ---Feature Importance

In [ ]:
def compute_and_plot_feature_importance(
    df,
    KU: float,
    KD: float,
    HOLD: int = HOLD,
    artifacts_dir: str = "artifacts",
    exclude_features=None,
    top_n: int = 20,
):
    if exclude_features is None:
        exclude_features = ['open', 'high', 'low', 'close', 'atr_200']

    artifacts_dir = Path(artifacts_dir)

    # 1) Split dataset
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
        df, KU, KD, HOLD=HOLD, force=False
    )

    X_train_f = X_train.drop(columns=exclude_features, errors='ignore')
    X_val_f   = X_val.drop(columns=exclude_features, errors='ignore')
    X_test_f  = X_test.drop(columns=exclude_features, errors='ignore')

    feature_names = list(X_train_f.columns)

    # 2) Loading models
    models = {}

    # CatBoost
    catboost_path = artifacts_dir / "cat" / f"best_catboost_ku{KU}_kd{KD}_hold{HOLD}.cbm"
    model_cat = CatBoostClassifier()
    model_cat.load_model(str(catboost_path))
    models["CatBoost"] = model_cat

    # LightGBM
    lgbm_path = artifacts_dir / "lgbm" / f"best_lgbm_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    model_lgb = joblib.load(lgbm_path)
    models["LightGBM"] = model_lgb

    # XGBoost
    xgb_path = artifacts_dir / "xgb" / f"best_xgb_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    model_xgb = joblib.load(xgb_path)
    models["XGBoost"] = model_xgb

    # RandomForest
    rf_path = artifacts_dir / "rf" / f"best_rf_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    model_rf = joblib.load(rf_path)
    models["RandomForest"] = model_rf

    # 3) Feature importance
    feature_importances = {}

    for name, model in models.items():
        try:
            # --- CatBoost ---
            if name == "CatBoost":
                imp = model.get_feature_importance(type="FeatureImportance")

            # --- XGBoost ---
            elif name == "XGBoost":
                booster = model.get_booster()
                score = booster.get_score(importance_type="gain")
                imp = np.zeros(len(feature_names))

                for key, val in score.items():

                    if key.startswith("f") and key[1:].isdigit():
                        idx = int(key[1:])
                        if idx < len(imp):
                            imp[idx] = val

                    elif key in feature_names:
                        idx = feature_names.index(key)
                        imp[idx] = val
                    else:
                        print(f"⚠️ XGB: skipping '{key}' (not in feature_names)")

            # --- LGBM / RF ---
            elif hasattr(model, "feature_importances_"):
                imp = model.feature_importances_

            else:
                print(f"⚠️ {name}: missing feature_importances_ → skip")
                continue


            if len(imp) != len(feature_names):
                print(f"⚠️ {name}: len(imp)={len(imp)} vs len(features)={len(feature_names)} → cutting")
                if len(imp) > len(feature_names):
                    imp = imp[:len(feature_names)]
                else:
                    imp = np.pad(imp, (0, len(feature_names)-len(imp)), constant_values=0)

            df_imp = pd.DataFrame({
                "Feature": feature_names,
                "Importance": imp
            })

            # normalization to [0,1] for convenience of comparison between models
            max_val = df_imp["Importance"].max()
            if max_val > 0:
                df_imp["Importance"] = df_imp["Importance"] / max_val

            feature_importances[name] = df_imp

        except Exception as e:
            print(f"⚠️ {name}: error reading importance → {e}")

    # 4) Merge all models into a single table
    merged = None
    for name, df_imp in feature_importances.items():
        df_imp = df_imp.rename(columns={"Importance": name})
        merged = df_imp if merged is None else merged.merge(df_imp, on="Feature", how="outer")

    merged = merged.fillna(0).set_index("Feature")

    # 5) Heatmap compare
    plt.figure(figsize=(12, 10))
    sns.heatmap(merged, cmap="YlGnBu")
    plt.title(f"Feature Importance Comparison\nKU={KU}, KD={KD}, HOLD={HOLD}")
    plt.xlabel("Model")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    # 6) Average importance and top-N
    merged["avg_importance"] = merged.mean(axis=1)
    df_top = merged.sort_values("avg_importance", ascending=False).head(top_n)

    plt.figure(figsize=(10, 8))
    plt.barh(df_top.index, df_top["avg_importance"])
    plt.gca().invert_yaxis()
    plt.title(f"Average Feature Importance (Top {top_n})\nKU={KU}, KD={KD}, HOLD={HOLD}")
    plt.xlabel("Average normalized importance")
    plt.tight_layout()
    plt.show()

    # 7) save in CSV
    out_csv = artifacts_dir / f"feature_importance_comparison_ku{KU}_kd{KD}_hold{HOLD}.csv"
    merged.to_csv(out_csv)
    print(f"✅ Save: {out_csv}")

    return merged


In [ ]:
fi_merged = compute_and_plot_feature_importance(
    df,
    KU=KU,
    KD=KD,
    HOLD=HOLD,
    artifacts_dir="artifacts",
    exclude_features=['open','high','low','close','atr_200'],
    top_n=23,
)


# ---ensemble_combinations (soft-voting)

In [ ]:
# ================================
# HELPERS
# ================================

def load_all_models(KU, KD, HOLD, artifacts_dir="artifacts"):
    models = {}
    '''
    # RF
    models["RF"] = joblib.load(
        f"{artifacts_dir}/rf/best_rf_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    )
    '''
    # LGBM
    models["LGBM"] = joblib.load(
        f"{artifacts_dir}/lgbm/best_lgbm_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    )

    # XGB
    models["XGB"] = joblib.load(
        f"{artifacts_dir}/xgb/best_xgb_ku{KU}_kd{KD}_hold{HOLD}.pkl"
    )

    # CAT
    model_cat = CatBoostClassifier()
    model_cat.load_model(
        f"{artifacts_dir}/cat/best_catboost_ku{KU}_kd{KD}_hold{HOLD}.cbm"
    )
    models["CAT"] = model_cat

    return models


# -----------------------------------------------
# Ensemble probability
# -----------------------------------------------
def ensemble_proba(models_list, proba_dict):
    """Compute soft-voting ensemble probability."""
    probs = [proba_dict[m] for m in models_list]
    return np.mean(probs, axis=0)


# -----------------------------------------------
# Evaluate ensemble with DL-style threshold search
# -----------------------------------------------
def evaluate_ensemble(models_list, proba_val, y_val, KU, KD):
    thr_list = np.linspace(0.1, 0.9, 33)

    best_thr = 0.5
    best_macro_f1 = -999
    best_metrics = None

    for thr in thr_list:
        y_pred = (proba_val >= thr).astype(int)

        m = triple_barrier_metrics(
            y_true=y_val,
            y_pred=y_pred,
            p_all=proba_val,
            ku=KU,
            kd=KD,
        )

        macro_f1 = m["macro_f1"]
        tp_prec = m["tp_precision"]
        BSS = m["bss"]

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_thr = thr
            best_metrics = m
        elif macro_f1 == best_macro_f1:
            if tp_prec > best_metrics["tp_precision"]:
                best_thr = thr
                best_metrics = m
            elif tp_prec == best_metrics["tp_precision"] and BSS > best_metrics["bss"]:
                best_thr = thr
                best_metrics = m

    return best_thr, best_metrics


# ===================================================
# MAIN: run ensemble evaluation
# ===================================================

def run_ensemble_models(df, KU, KD, HOLD=96):
    print(f"\n=== ENSEMBLES for KU={KU}, KD={KD}, HOLD={HOLD} ===")

    # Load split
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
        df, KU, KD, HOLD=HOLD, force=False
    )

    # Load all models
    models = load_all_models(KU, KD, HOLD)

    # Compute proba for each model
    proba_val = {}
    proba_test = {}
    for name, model in models.items():
        proba_val[name] = model.predict_proba(X_val)[:, 1]
        proba_test[name] = model.predict_proba(X_test)[:, 1]

    # Generate all 2-, 3-, and 4-model ensembles
    model_names = list(models.keys())
    all_combos = []

    for r in [2, 3, 4]:
        combos = list(itertools.combinations(model_names, r))
        all_combos.extend(combos)

    records = []

    for combo in all_combos:
        combo_name = "+".join(combo)

        # Ensemble proba (soft voting)
        p_val = ensemble_proba(combo, proba_val)
        p_test = ensemble_proba(combo, proba_test)

        # Best threshold (DL-style)
        best_thr, m_val = evaluate_ensemble(combo, p_val, y_val, KU, KD)

        # Apply on test
        y_pred_test = (p_test >= best_thr).astype(int)
        m_test = triple_barrier_metrics(
            y_true=y_test,
            y_pred=y_pred_test,
            p_all=p_test,
            ku=KU,
            kd=KD
        )

        records.append({
            "ensemble": combo_name,
            "threshold": best_thr,
            "macro_f1_val": m_val["macro_f1"],
            "macro_f1_test": m_test["macro_f1"],
            "tp_prec_test": m_test["tp_precision"],
            "tp_recall_test": m_test["tp_recall"],
            "BSS_test": m_test["bss"],
        })

        print(f" → {combo_name}: macro_f1_test={m_test['macro_f1']:.4f}")

    df_ens = pd.DataFrame(records)
    return df_ens


In [ ]:
df_ens = run_ensemble_models(df, KU=KU, KD=KD, HOLD=HOLD)
df_ens.sort_values("macro_f1_test", ascending=False)


# ---Ensemble (weighted ensemble)

In [ ]:
def weighted_stacking_v3(df, KU, KD, HOLD=96, n_trials=60):

    print("\n============== Weighted Stacking v3 ==============\n")

    # --- Load split ---
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
        df, KU, KD, HOLD=HOLD, force=False
    )

    y_train = np.asarray(y_train)
    y_val   = np.asarray(y_val)
    y_test  = np.asarray(y_test)

    # Load saved models
    models = {
        #"rf":  joblib.load(f"artifacts/rf/best_rf_ku{KU}_kd{KD}_hold{HOLD}.pkl"),
        "lgb": joblib.load(f"artifacts/lgbm/best_lgbm_ku{KU}_kd{KD}_hold{HOLD}.pkl"),
        "xgb": joblib.load(f"artifacts/xgb/best_xgb_ku{KU}_kd{KD}_hold{HOLD}.pkl"),
    }

    cat = CatBoostClassifier()
    cat.load_model(f"artifacts/cat/best_catboost_ku{KU}_kd{KD}_hold{HOLD}.cbm")
    models["cat"] = cat

    # Precompute probabilities
    val_preds = {name: m.predict_proba(X_val)[:, 1] for name, m in models.items()}
    test_preds = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}

    # ---- Optuna objective ----
    def objective(trial):

        #w_rf  = trial.suggest_float("w_rf",  0, 1)
        w_lgb = trial.suggest_float("w_lgb", 0, 1)
        w_xgb = trial.suggest_float("w_xgb", 0, 1)
        w_cat = trial.suggest_float("w_cat", 0, 1)

        # Normalize weights
        s = w_lgb + w_xgb + w_cat
        if s == 0:
            return -999

        w_lgb, w_xgb, w_cat = [w_lgb/s, w_xgb/s, w_cat/s]

        p = (
            w_lgb * val_preds["lgb"] +
            w_xgb * val_preds["xgb"] +
            w_cat * val_preds["cat"]
        )

        y_pred = (p >= 0.5).astype(int)

        m = triple_barrier_metrics(
            y_true=y_val,
            y_pred=y_pred,
            p_all=p,
            ku=KU, kd=KD
        )

        # DL-style objective
        score = (
            m["macro_f1"] * 1.0 +
            m["tp_precision"] * 0.001 +
            m["bss"] * 0.0001
        )

        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    best_w = study.best_params
    print("\n🎯 Best weights found:", best_w)

    # ---- FINAL TEST ----
    w_lgb, w_xgb, w_cat = best_w.values()
    s = w_lgb + w_xgb + w_cat
    w_lgb, w_xgb, w_cat = [w_lgb/s, w_xgb/s, w_cat/s]

    p_test_final = (
        w_lgb * test_preds["lgb"] +
        w_xgb * test_preds["xgb"] +
        w_cat * test_preds["cat"]
    )

    y_pred_test = (p_test_final >= 0.5).astype(int)

    m_test = triple_barrier_metrics(
        y_true=y_test,
        y_pred=y_pred_test,
        p_all=p_test_final,
        ku=KU, kd=KD
    )

    print("\n🔥 FINAL Weighted Stacking v3 RESULTS:")
    for k, v in m_test.items():
        print(f"{k}: {v}")

    return best_w, m_test


In [ ]:
weights, ensemble_metrics = weighted_stacking_v3(
    df,
    KU=KU,
    KD=KD,
    HOLD=HOLD,
    n_trials=60
)


# ---Ensemble (meta-learner with using KFold OOF predictions for meta-training)

In [ ]:
def stacking_v2_oof(df, KU=KU, KD=KD, HOLD=HOLD, n_splits=5):
    """
    OOF-stacking v2:
      - OOF по TRAIN
      - meta-модель (LGBM) вчиться на OOF
      - VAL/TEST перетворюються в meta-фічі (4 ймовірності)
      - повертає:
          meta_model,
          base_models,
          meta_val (VAL meta features),
          y_val,
          meta_test (TEST meta features),
          y_test,
          сирі метрики meta-моделі на TEST (thr=0.5)
    """

    print("\n===== RUN STACKING v2 (OOF) =====")

    # 1) split
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
        df, KU=KU, KD=KD, HOLD=HOLD, force=False
    )

    y_train = y_train.values.ravel()
    y_val   = y_val.values.ravel()
    y_test  = y_test.values.ravel()

    # 2) upload models
    #model_rf   = joblib.load(f"artifacts/rf/best_rf_ku{KU}_kd{KD}_hold{HOLD}.pkl")
    model_lgbm = joblib.load(f"artifacts/lgbm/best_lgbm_ku{KU}_kd{KD}_hold{HOLD}.pkl")
    model_xgb  = joblib.load(f"artifacts/xgb/best_xgb_ku{KU}_kd{KD}_hold{HOLD}.pkl")

    model_cat = CatBoostClassifier()
    model_cat.load_model(f"artifacts/cat/best_catboost_ku{KU}_kd{KD}_hold{HOLD}.cbm")

    base_models = {
        #"rf": model_rf,
        "lgb": model_lgbm,
        "xgb": model_xgb,
        "cat": model_cat,
    }

    model_names = list(base_models.keys())
    n_models = len(model_names)

    # 3) OOF-matrix for TRAIN
    oof_preds = np.zeros((len(X_train), n_models))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_train)):
        print(f"\n----- Fold {fold+1}/{n_splits} -----")
        X_tr, y_tr = X_train.iloc[tr_idx], y_train[tr_idx]
        X_vl, y_vl = X_train.iloc[vl_idx], y_train[vl_idx]

        for j, name in enumerate(model_names):
            original = base_models[name]

            # ---- cloning ----
            if name == "cat":
                params = original.get_params()
                params.pop("verbose", None)
                model = CatBoostClassifier(**params)
                model.set_params(verbose=False)
            else:
                model = deepcopy(original)

                if isinstance(model, XGBClassifier):
                    if "early_stopping_rounds" in model.get_params():
                        model.set_params(early_stopping_rounds=None)

                if isinstance(model, LGBMClassifier):
                    if "callbacks" in model.get_params():
                        model.set_params(callbacks=None)

            model.fit(X_tr, y_tr)
            p = model.predict_proba(X_vl)[:, 1]
            oof_preds[vl_idx, j] = p

            print(f"  {name} OOF done.")

    # 4) meta-learner
    meta_model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=15,
        random_state=42,
    )

    print("\nTraining meta-learner on OOF...")
    meta_model.fit(oof_preds, y_train)

    # 5) meta-features for VAL and TEST
    meta_val = np.column_stack([
        #base_models["rf"].predict_proba(X_val)[:, 1],
        base_models["lgb"].predict_proba(X_val)[:, 1],
        base_models["xgb"].predict_proba(X_val)[:, 1],
        base_models["cat"].predict_proba(X_val)[:, 1],
    ])

    meta_test = np.column_stack([
        #base_models["rf"].predict_proba(X_test)[:, 1],
        base_models["lgb"].predict_proba(X_test)[:, 1],
        base_models["xgb"].predict_proba(X_test)[:, 1],
        base_models["cat"].predict_proba(X_test)[:, 1],
    ])

    # 6) raw meta-prediction on TEST
    proba_test_raw = meta_model.predict_proba(meta_test)[:, 1]
    y_pred_test_raw = (proba_test_raw >= 0.5).astype(int)

    meta_metrics_raw = triple_barrier_metrics(
        y_true=y_test,
        y_pred=y_pred_test_raw,
        p_all=proba_test_raw,
        ku=KU,
        kd=KD,
    )

    print("\n🔥 STACKING v2 (raw, thr=0.5) on TEST:")
    for k, v in meta_metrics_raw.items():
        print(f"{k}: {v:.6f}")

    return {
        "meta_model": meta_model,
        "base_models": base_models,
        "meta_val": meta_val,
        "y_val": y_val,
        "meta_test": meta_test,
        "y_test": y_test,
        "meta_metrics_raw": meta_metrics_raw,
    }


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def calibrate_meta_platt(meta_model, meta_val, y_val, meta_test, y_test, KU, KD):
    """
    Calibrate the meta_model on meta_val using Platt scaling (sigmoid),
    compute the metrics on TEST, and also the DL-threshold for the calibrated probabilities.
    """

    print("\n===== Calibrating META (Platt / sigmoid) =====")

    cal = CalibratedClassifierCV(meta_model, method="sigmoid", cv="prefit")
    cal.fit(meta_val, y_val)

    # calibrated probabilities on TEST
    proba_test_cal = cal.predict_proba(meta_test)[:, 1]
    y_pred_test_cal_05 = (proba_test_cal >= 0.5).astype(int)

    metrics_cal_05 = triple_barrier_metrics(
        y_true=y_test,
        y_pred=y_pred_test_cal_05,
        p_all=proba_test_cal,
        ku=KU,
        kd=KD,
    )

    print("\n🔥 Calibrated META (thr=0.5) TEST metrics:")
    for k, v in metrics_cal_05.items():
        print(f"{k}: {v:.6f}")

    # DL-threshold on top of the calibrated probabilities
    best_thr, best_metrics = dl_threshold_for_probs(
        proba=proba_test_cal,
        y_true=y_test,
        KU=KU,
        KD=KD,
    )

    print("\n🔥 Calibrated META + DL-threshold TEST metrics:")
    for k, v in best_metrics.items():
        print(f"{k}: {v:.6f}")

    return {
        "calibrator": cal,
        "proba_test_cal": proba_test_cal,
        "metrics_thr_05": metrics_cal_05,
        "best_thr": best_thr,
        "metrics_best_thr": best_metrics,
    }


In [ ]:
# 1) Stacking v2 + OOF + meta-features
stack_res = stacking_v2_oof(df, KU=KU, KD=KD, HOLD=HOLD, n_splits=5)

meta_model   = stack_res["meta_model"]
base_models  = stack_res["base_models"]
meta_val     = stack_res["meta_val"]
y_val        = stack_res["y_val"]
meta_test    = stack_res["meta_test"]
y_test       = stack_res["y_test"]
meta_raw_met = stack_res["meta_metrics_raw"]


In [ ]:

# 2) Calibrate the meta model + compute metrics
calib_res = calibrate_meta_platt(
    meta_model=meta_model,
    meta_val=meta_val,
    y_val=y_val,
    meta_test=meta_test,
    y_test=y_test,
    KU=KU,
    KD=KD,
)


In [ ]:
def evaluate_meta_calibrated(cal_model, X_test, y_test, KU, KD, thr):
    proba = cal_model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= thr).astype(int)

    m = triple_barrier_metrics(
        y_true=y_test,
        y_pred=y_pred,
        p_all=proba,
        ku=KU,
        kd=KD
    )

    return {
        "macro_f1": m["macro_f1"],
        "tp_precision": m["tp_precision"],
        "tp_recall": m["tp_recall"],
        "bss": m["bss"]
    }


In [ ]:
def calibrate_meta(meta_model, X_val, y_val):
    calibrated = CalibratedClassifierCV(meta_model, method="sigmoid", cv="prefit")
    calibrated.fit(X_val, y_val)
    return calibrated

In [ ]:
# 1. calibration
cal_meta = calibrate_meta(meta_model, meta_val, y_val)
proba_test_cal = cal_meta.predict_proba(meta_test)[:, 1]

# 2. threshold search
proba_meta_val = cal_meta.predict_proba(meta_val)[:, 1]

best_meta_thr, meta_thr_metrics = dl_threshold_for_probs(
    proba_meta_val, y_val, KU=KU, KD=KD
)

# 3. test eval
proba_meta_test = cal_meta.predict_proba(meta_test)[:, 1]
y_pred_meta_test = (proba_meta_test >= best_meta_thr).astype(int)

meta_calibrated_metrics = triple_barrier_metrics(
    y_true=y_test,
    y_pred=y_pred_meta_test,
    p_all=proba_meta_test,
    ku=KU,
    kd=KD
)

meta_calibrated_metrics




In [ ]:
print(X_test.columns)

# ---model calcalibration

In [ ]:
def eval_from_proba(name, proba_val, proba_test, y_val, y_test, KU, KD):
    """
    1) Selects the DL-threshold on validation
    2) Computes the metrics on TEST using this threshold
    """
    best_thr, best_val_metrics = dl_threshold_for_probs(
        proba_val, y_val, KU=KU, KD=KD
    )

    y_pred_test = (proba_test >= best_thr).astype(int)

    m_test = triple_barrier_metrics(
        y_true=y_test,
        y_pred=y_pred_test,
        p_all=proba_test,
        ku=KU,
        kd=KD,
    )

    res = {
        "model": name,
        "threshold": best_thr,
        "macro_f1_val":  best_val_metrics["macro_f1"],
        "tp_precision_val": best_val_metrics["tp_precision"],
        "tp_recall_val":    best_val_metrics["tp_recall"],
        "tp_f1_val":        best_val_metrics["tp_f1"],
        "BSS_val":          best_val_metrics["bss"],
        "macro_f1_test": m_test["macro_f1"],
        "tp_precision_test": m_test["tp_precision"],
        "tp_recall_test":    m_test["tp_recall"],
        "tp_f1_test":        m_test["tp_f1"],
        "BSS_test":          m_test["bss"],
    }
    return res


# --- base models calibration

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def calibrate_all_base_models(df, KU, KD, HOLD=96, artifacts_dir="artifacts"):

    X_train, y_train, X_val, y_val, X_test, y_test = prepare_split_for_ku_kd(
        df, KU=KU, KD=KD, HOLD=HOLD, force=False
    )

    y_val  = y_val.values.ravel()
    y_test = y_test.values.ravel()

    rf   = joblib.load(f"{artifacts_dir}/rf/best_rf_ku{KU}_kd{KD}_hold{HOLD}.pkl")
    lgbm = joblib.load(f"{artifacts_dir}/lgbm/best_lgbm_ku{KU}_kd{KD}_hold{HOLD}.pkl")
    xgb  = joblib.load(f"{artifacts_dir}/xgb/best_xgb_ku{KU}_kd{KD}_hold{HOLD}.pkl")
    cat  = CatBoostClassifier()
    cat.load_model(f"{artifacts_dir}/cat/best_catboost_ku{KU}_kd{KD}_hold{HOLD}.cbm")

    base_models = {
        "RF": rf,
        "LGBM": lgbm,
        "XGB": xgb,
        "CAT": cat,
    }

    raw_results = []
    cal_results = []

    # Dictionaries of probabilities, so that ensembles can be built later if desired
    proba_raw_val  = {}
    proba_raw_test = {}
    proba_cal_val  = {}
    proba_cal_test = {}

    for name, model in base_models.items():
        print(f"\n===== {name} (raw) =====")

        # RAW proba
        proba_val  = model.predict_proba(X_val)[:, 1]
        proba_test = model.predict_proba(X_test)[:, 1]

        proba_raw_val[name]  = proba_val
        proba_raw_test[name] = proba_test

        # RAW metrics
        raw_res = eval_from_proba(
            name + "_RAW",
            proba_val,
            proba_test,
            y_val,
            y_test,
            KU,
            KD,
        )
        raw_results.append(raw_res)

        # === Calibrating models (Platt, cv='prefit') ===
        print(f"Calibrating {name} with Platt scaling...")
        cal = CalibratedClassifierCV(model, method="sigmoid", cv="prefit")
        cal.fit(X_val, y_val)

        proba_val_cal  = cal.predict_proba(X_val)[:, 1]
        proba_test_cal = cal.predict_proba(X_test)[:, 1]

        proba_cal_val[name]  = proba_val_cal
        proba_cal_test[name] = proba_test_cal

        cal_res = eval_from_proba(
            name + "_CAL",
            proba_val_cal,
            proba_test_cal,
            y_val,
            y_test,
            KU,
            KD,
        )
        cal_results.append(cal_res)

    df_raw = pd.DataFrame(raw_results)
    df_cal = pd.DataFrame(cal_results)

    return df_raw, df_cal, proba_raw_val, proba_raw_test, proba_cal_val, proba_cal_test


In [ ]:
df_base_raw, df_base_cal, proba_raw_val, proba_raw_test, proba_cal_val, proba_cal_test = calibrate_all_base_models(
    df, KU=KU, KD=KD, HOLD=HOLD
)

df_base_raw, df_base_cal


# --- save predictions

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. RAW TEST SET + y_true
# ============================================================
raw_test = pd.read_parquet("artifacts/splits/ku6.0_kd2.0_hold336/test.parquet")
#raw_test_trimmed = raw_test.iloc[336:].reset_index(drop=True)

# ground truth (отриманий із prepare_split_for_ku_kd)
y_true = y_test


# ============================================================
# 2. SAVE RAW/CALIBRATED/STACKING PREDICTIONS
# ============================================================
def save_trade_predictions_from_proba(proba, model_name):
    """
    Saves a CSV file with the model's predictions:
    - Input: the array of probabilities proba after calibration/stacking/raw
    - Output: a CSV in artifacts/
    """
    df_out = pd.DataFrame({
        "hour_sin":      raw_test["hour_sin"].values,
        "hour_cos":      raw_test["hour_cos"].values,
        "dayofweek_sin": raw_test["dayofweek_sin"].values,
        "dayofweek_cos": raw_test["dayofweek_cos"].values,
        "close":         raw_test["close"].values,
        "atr_200":       raw_test["atr_200"].values,
        "p_model":       proba,
        "y_true":        y_true,
        "y_pred":        (proba >= 0.5).astype(int),
    })

    fname = f"artifacts/trade_predictions_{model_name}.csv"
    df_out.to_csv(fname, index=False)
    print(f"[OK] Saved {fname}")

    return df_out


# ============================================================
# 3. OPTION WITH A CUSTOM THRESHOLD (DL-threshold)
# ============================================================
def save_trade_predictions_with_threshold(proba, thr, model_name):
    """
    Saves the prediction with the threshold thr (e.g., the DL-threshold)
    """
    df_out = pd.DataFrame({
        "hour_sin":      raw_test["hour_sin"].values,
        "hour_cos":      raw_test["hour_cos"].values,
        "dayofweek_sin": raw_test["dayofweek_sin"].values,
        "dayofweek_cos": raw_test["dayofweek_cos"].values,
        "close":         raw_test["close"].values,
        "atr_200":       raw_test["atr_200"].values,
        "p_model":       proba,
        "y_true":        y_true,
        "y_pred":        (proba >= thr).astype(int),
    })

    fname = f"artifacts/trade_predictions_{model_name}.csv"
    df_out.to_csv(fname, index=False)
    print(f"[OK] Saved {fname}")

    return df_out

# --- Generating stacking predictions ---
proba_test_raw = meta_model.predict_proba(meta_test)[:, 1]

# DL threshold over calibrated meta
best_thr, _ = dl_threshold_for_probs(
    proba_meta_val,
    y_val,
    KU=KU,
    KD=KD
)


In [ ]:
# ============================================================
#  RECALCULATE METRICS (CLEAN VERSION)
# ============================================================
all_results = []

def add_metrics(model_name, proba, thr=0.5):
    y_pred = (proba >= thr).astype(int)
    m = triple_barrier_metrics(
        y_true=y_true,
        y_pred=y_pred,
        p_all=proba,
        ku=KU,
        kd=KD
    )
    m["model"] = model_name
    m["threshold"] = thr
    all_results.append(m)


# Base RAW models
add_metrics("RF_raw",   proba_raw_test["RF"])
add_metrics("LGBM_raw", proba_raw_test["LGBM"])
add_metrics("XGB_raw",  proba_raw_test["XGB"])
add_metrics("CAT_raw",  proba_raw_test["CAT"])

# Stacking
add_metrics("stacking_raw",        proba_test_raw)
add_metrics("stacking_calibrated", proba_test_cal, thr=best_thr)


# SAVE METRIC TABLE
df_metric_results = pd.DataFrame(all_results)
df_metric_results.to_excel("artifacts/all_metrics_clean.xlsx", index=False)

print(df_metric_results)

In [ ]:
# ============================================================
# 4. SAVE PREDICTIONS FOR ALL MODELS
#    (RAW + CALIBRATED + STACKING RAW + STACKING CAL)
# ============================================================

# --- RAW base models ---
save_trade_predictions_from_proba(proba_raw_test["RF"],   "rf_raw")
save_trade_predictions_from_proba(proba_raw_test["LGBM"], "lgbm_raw")
save_trade_predictions_from_proba(proba_raw_test["XGB"],  "xgb_raw")
save_trade_predictions_from_proba(proba_raw_test["CAT"],  "cat_raw")

'''
# --- CALIBRATED base models ---
save_trade_predictions_from_proba(proba_cal_test["RF"],   "rf_cal")
save_trade_predictions_from_proba(proba_cal_test["LGBM"], "lgbm_cal")
save_trade_predictions_from_proba(proba_cal_test["XGB"],  "xgb_cal")
save_trade_predictions_from_proba(proba_cal_test["CAT"],  "cat_cal")
'''

# --- STACKING RAW ---
#save_trade_predictions_from_proba(proba_test_raw, "stacking_raw")

# --- STACKING CALIBRATED + BEST THRESHOLD ---
save_trade_predictions_with_threshold(
    proba_test_cal,
    best_thr,
    "stacking_calibrated"
)

# --- ensemble calcalibration


In [ ]:
def ensemble_proba(models, proba_dict):
    """
    models: list of model names ["RF", "CAT"]
    proba_dict: dictionary {"RF": ..., "CAT": ...} with probabilities
    """
    arr = [proba_dict[m] for m in models]
    return np.mean(arr, axis=0)


In [ ]:
import itertools

def generate_all_combinations(model_names):
    combos = []
    for r in [2, 3, 4]:
        for comb in itertools.combinations(model_names, r):
            combos.append(list(comb))
    return combos


In [ ]:
def evaluate_all_ensembles(
    proba_raw_val, proba_raw_test,
    proba_cal_val, proba_cal_test,
    y_val, y_test,
    KU, KD
):
    model_names = list(proba_raw_val.keys())
    combos = generate_all_combinations(model_names)

    records = []

    for comb in combos:
        name = "+".join(comb)

        # -------------------------------------------------------------
        # RAW PROBAS
        # -------------------------------------------------------------
        proba_val_raw  = ensemble_proba(comb, proba_raw_val)
        proba_test_raw = ensemble_proba(comb, proba_raw_test)

        raw_thr, raw_m = dl_threshold_for_probs(
            proba_val_raw, y_val, KU, KD
        )

        y_pred_raw = (proba_test_raw >= raw_thr).astype(int)
        test_raw_m = triple_barrier_metrics(
            y_true=y_test,
            y_pred=y_pred_raw,
            p_all=proba_test_raw,
            ku=KU, kd=KD
        )

        # valid F1
        tp_f1_val = (
            2 * raw_m["tp_precision"] * raw_m["tp_recall"] /
            (raw_m["tp_precision"] + raw_m["tp_recall"])
            if (raw_m["tp_precision"] + raw_m["tp_recall"]) > 0 else None
        )

        tp_f1_test = (
            2 * test_raw_m["tp_precision"] * test_raw_m["tp_recall"] /
            (test_raw_m["tp_precision"] + test_raw_m["tp_recall"])
            if (test_raw_m["tp_precision"] + test_raw_m["tp_recall"]) > 0 else None
        )

        records.append({
            "model": None,
            "ensemble": name + "_RAW",

            "threshold_used": raw_thr,
            "threshold": raw_thr,

            "macro_f1_val": raw_m["macro_f1"],
            "tp_precision_val": raw_m["tp_precision"],
            "tp_recall_val": raw_m["tp_recall"],
            "tp_f1_val": tp_f1_val,
            "BSS_val": raw_m["bss"],

            "macro_f1_test": test_raw_m["macro_f1"],
            "tp_precision_test": test_raw_m["tp_precision"],
            "tp_recall_test": test_raw_m["tp_recall"],
            "tp_f1_test": tp_f1_test,
            "BSS_test": test_raw_m["bss"]
        })

        # -------------------------------------------------------------
        # CALIBRATED PROBAS
        # -------------------------------------------------------------
        proba_val_cal  = ensemble_proba(comb, proba_cal_val)
        proba_test_cal = ensemble_proba(comb, proba_cal_test)

        cal_thr, cal_m = dl_threshold_for_probs(
            proba_val_cal, y_val, KU, KD
        )

        y_pred_cal = (proba_test_cal >= cal_thr).astype(int)
        test_cal_m = triple_barrier_metrics(
            y_true=y_test,
            y_pred=y_pred_cal,
            p_all=proba_test_cal,
            ku=KU, kd=KD
        )

        tp_f1_val = (
            2 * cal_m["tp_precision"] * cal_m["tp_recall"] /
            (cal_m["tp_precision"] + cal_m["tp_recall"])
        ) if (cal_m["tp_precision"] + cal_m["tp_recall"]) > 0 else None

        tp_f1_test = (
            2 * test_cal_m["tp_precision"] * test_cal_m["tp_recall"] /
            (test_cal_m["tp_precision"] + test_cal_m["tp_recall"])
        ) if (test_cal_m["tp_precision"] + test_cal_m["tp_recall"]) > 0 else None

        records.append({
            "ensemble": name + "_CAL",

            "threshold_used": cal_thr,

            "macro_f1_val": cal_m["macro_f1"],
            "tp_precision_val": cal_m["tp_precision"],
            "tp_recall_val": cal_m["tp_recall"],
            "tp_f1_val": tp_f1_val,
            "BSS_val": cal_m["bss"],

            "macro_f1_test": test_cal_m["macro_f1"],
            "tp_precision_test": test_cal_m["tp_precision"],
            "tp_recall_test": test_cal_m["tp_recall"],
            "tp_f1_test": tp_f1_test,
            "BSS_test": test_cal_m["bss"]
        })


    df_ens = pd.DataFrame(records)

    # Reordering columns
    ordered_cols = [
        "ensemble",
        "threshold_used",

        "macro_f1_val",
        "tp_precision_val", "tp_recall_val", "tp_f1_val", "BSS_val",

        "macro_f1_test",
        "tp_precision_test", "tp_recall_test", "tp_f1_test", "BSS_test",
    ]

    return df_ens[ordered_cols]


In [ ]:
df_ens_full = evaluate_all_ensembles(
    proba_raw_val, proba_raw_test,
    proba_cal_val, proba_cal_test,
    y_val, y_test,
    KU=KU, KD=KD
)

df_ens_full


In [ ]:
extra_rows = []

# META_CALIBRATED (if there is calib_res)
if "calib_res" in globals():
    m = calib_res["metrics_best_thr"]
    extra_rows.append({
        "ensemble": "META_CALIBRATED",
        "threshold": calib_res["best_thr"] if "best_thr" in calib_res else np.nan,
        "macro_f1_val": np.nan,
        "tp_precision_val": np.nan,
        "tp_recall_val": np.nan,
        "tp_f1_val": np.nan,
        "BSS_val": np.nan,
        "macro_f1_test": m["macro_f1"],
        "tp_precision_test": m["tp_precision"],
        "tp_recall_test": m["tp_recall"],
        "tp_f1_test": m["tp_f1"],
        "BSS_test": m["bss"],
    })

# WEIGHTED_v3 (if there are ensemble_metrics from the stack)
if "ensemble_metrics" in globals():
    em = ensemble_metrics
    extra_rows.append({
        "ensemble": "WEIGHTED_STACKING",
        "threshold": np.nan,
        "macro_f1_val": np.nan,
        "tp_precision_val": np.nan,
        "tp_recall_val": np.nan,
        "tp_f1_val": np.nan,
        "BSS_val": np.nan,
        "macro_f1_test": em["macro_f1"],
        "tp_precision_test": em["tp_precision"],
        "tp_recall_test": em["tp_recall"],
        "tp_f1_test": em["tp_f1"],
        "BSS_test": em["bss"],
    })

df_extra = pd.DataFrame(extra_rows) if extra_rows else pd.DataFrame()


In [ ]:
df_extra = pd.DataFrame(extra_rows) if extra_rows else pd.DataFrame()
df_extra

In [ ]:
import pandas as pd
from pathlib import Path

def normalize_columns(df):
    """Bringing all column names to a single standard"""
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    
    col_map = {
        "ensemble": "model",
        "model_name": "model",
        "thr": "threshold_used",
        "threshold": "threshold_used",

        "tp_prec_test": "tp_precision_test",
        "tp_precision": "tp_precision_test",
        "precision_test": "tp_precision_test",

        "tp_prec_val": "tp_precision_val",
        "precision_val": "tp_precision_val",

        "recall_test": "tp_recall_test",
        "tp_rec_test": "tp_recall_test",

        "recall_val": "tp_recall_val",
        "tp_rec_val": "tp_recall_val",

        "bss": "bss_test",
        "bss_val": "bss_val",
        "bss_test": "bss_test",
    }
    
    df = df.rename(columns=col_map)
    return df


def merge_duplicate_cols(df):
    """Merges duplicate columns: _x, _y → a single column"""
    fixed = {}
    for col in df.columns:
        base = col.rstrip("_xy")
        if base not in fixed:
            fixed[base] = df[col]
        else:
            fixed[base] = fixed[base].fillna(df[col])
    return pd.DataFrame(fixed)


def build_leaderboard(df_base_raw, df_base_cal, df_ens_full, df_extra, KU, KD, HOLD):
    print("📌 Normalizing columns...")

    dfs = []

    # normalize
    if df_base_raw is not None and not df_base_raw.empty:
        dfs.append(normalize_columns(df_base_raw))

    if df_base_cal is not None and not df_base_cal.empty:
        dfs.append(normalize_columns(df_base_cal))

    if df_ens_full is not None and not df_ens_full.empty:
        dfs.append(normalize_columns(df_ens_full))

    if df_extra is not None and not df_extra.empty:
        dfs.append(normalize_columns(df_extra))

    print(f"📌 Combining {len(dfs)} result tables...")

    dfs = [df.reset_index(drop=True) for df in dfs]
    
    # merge
    df_leaderboard = pd.concat(dfs, ignore_index=True)

    print("📌 Merging duplicate columns...")
    df_leaderboard = merge_duplicate_cols(df_leaderboard)

    # remove all val-metrics
    val_cols = [c for c in df_leaderboard.columns if c.endswith("_val")]
    if val_cols:
        print(" Removing validation columns:", val_cols)
        df_leaderboard = df_leaderboard.drop(columns=val_cols)

    # check core columns
    needed_cols = [
        "model", "threshold_used",
        "macro_f1_test", "tp_precision_test", "tp_recall_test", 
        "tp_f1_test", "bss_test"
    ]
    for col in needed_cols:
        if col not in df_leaderboard.columns:
            df_leaderboard[col] = None

    # --- Sorting ---
    df_sorted = df_leaderboard.sort_values(
        ["macro_f1_test", "tp_precision_test", "bss_test"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    # --- Save ---
    save_dir = Path("artifacts") / "results_binary"
    save_dir.mkdir(parents=True, exist_ok=True)
    leaderboard_path = save_dir / f"leaderboard_ku{KU}_kd{KD}_hold{HOLD}.xlsx"

    df_sorted.to_excel(leaderboard_path, index=False)

    print("✅ Leaderboard saved to:", leaderboard_path)
    return df_sorted


In [ ]:
df_leaderboard_sorted = build_leaderboard(
    df_base_raw,
    df_base_cal,
    df_ens_full,
    df_extra,     
    KU=KU,
    KD=KD,
    HOLD=HOLD
)

df_leaderboard_sorted.head(10)


In [ ]:
# add SUPER META CALIBRATED
df_compare = df_ens.copy()

df_compare.loc[len(df_compare)] = [
    "META_CALIBRATED",
    None,  # threshold
    None,  # macro_f1_val
    meta_calibrated_metrics["macro_f1"],
    meta_calibrated_metrics["tp_precision"],
    meta_calibrated_metrics["tp_recall"],
    meta_calibrated_metrics["bss"]
]

print(df_compare)
